In [1]:
# Setup: Import modules and define paths
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Now import validation functions
from functions.validation_functions import (
    load_groundwater_csv,
    flag_physical_bounds,
    flag_unrealistic_step_change,
    flag_constant_head_periods,
    flag_statistical_outliers,
)

# Dataset roots (per-origin structure)
wiertsema_input_dir = repo_root / 'output_data' / 'wiertsema'
fugro_input_dir = repo_root / 'output_data' / 'fugro'

print('Setup complete!')
print(f'Repo root: {repo_root}')
print(f'Wiertsema dataset root: {wiertsema_input_dir}')
print(f'Fugro dataset root: {fugro_input_dir}')

Setup complete!
Repo root: d:\Users\jvanruitenbeek\data_validation
Wiertsema dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
Fugro dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\fugro


# Batch Processing (All Files)


In [2]:
# Step 1: Define validation parameters (adjust these based on your data)

# Physical bounds (meters NAP or similar datum)
#hmin = -10  # Minimum realistic head value
#hmax = 10   # Maximum realistic head value #
# CURRENTLY DEFINED IN LOOP

# Maximum allowed rate of change between timestamps in m/day
max_up = 0.3
max_down = -0.05  

# Constant head periods
tconst_steps = 24   # Flag if at least 24 hourly measurements are constant
flat_margin_m = 0.02  # Checking the 'dH' column for changes smaller than this
min_band_m = 0.15   # Using the minimum value from the head series, then adding this margin on top


# print('Validation parameters set:')
# print(f'  Physical bounds: {hmin} to {hmax} m')
#print(f'  Constant head: >{tconst_days} days or >{nconst_min} measurements')

In [3]:
# Configuration: Choose which dataset to process
# Options: 'wiertsema' or 'fugro'
folder_choice = 'wiertsema'

# Select dataset root based on choice
if folder_choice.lower() == 'wiertsema':
    input_root = wiertsema_input_dir
    print('Selected: WIERTSEMA')
elif folder_choice.lower() == 'fugro':
    input_root = fugro_input_dir
    print('Selected: FUGRO')
else:
    raise ValueError("folder_choice must be 'wiertsema' or 'fugro'")

print('   Batch Processing Configuration:')
print(f"  Dataset root: {input_root}")
print("  Input pattern: <origin>/knmi/*.csv")
print("  Output pattern: <origin>/validated/*.csv")
print(f"  Other parameters: tconst_steps={tconst_steps}, flat_margin_m={flat_margin_m}, min_band_m={min_band_m}")

Selected: WIERTSEMA
   Batch Processing Configuration:
  Dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
  Input pattern: <origin>/knmi/*.csv
  Output pattern: <origin>/validated/*.csv
  Other parameters: tconst_steps=24, flat_margin_m=0.02, min_band_m=0.15


In [4]:
# Loading object data
object_data = pd.read_csv(repo_root / 'output_data' / 'object_data.csv', index_col=0)

In [5]:
# Load object data ONCE before the loop (better & faster)
object_data = pd.read_csv(repo_root / "output_data" / "object_data.csv", index_col=0)

print("object_data index sample (first 5 rows):")
for idx in list(object_data.index[:5]):
    print("  ", repr(idx))
print()

# Execute batch processing on per-origin KNMI files
csv_files = sorted(input_root.glob('*/knmi/*.csv'))
print(f"Found {len(csv_files)} CSV files to process\n")

results = []
failed_files = []

for i, csv_file in enumerate(csv_files, 1):
    source_origin_stem = csv_file.parent.parent.name
    try:
        print(f"[{i}/{len(csv_files)}] Processing: {csv_file.name} (origin={source_origin_stem})")

        # Load sensor data
        df = load_groundwater_csv(csv_file)
        initial_rows = len(df)
        print(f"  Initial rows: {initial_rows}")

        # Persist 3-day and 7-day rolling means computed from the UNVALIDATED
        # head_raw column. These columns survive the v1-v4 flagging steps
        # untouched and are written into the validated CSV.
        df = df.set_index("Time", drop=False).sort_index()
        df["head_raw_3d"] = df["head_raw"].rolling("3D", min_periods=1).mean()
        df["head_raw_7d"] = df["head_raw"].rolling("7D", min_periods=1).mean()
        df = df.reset_index(drop=True)

        # Defaults; optionally overridden from object_data.csv
        hmin = -10
        hmax = 10

        fname = csv_file.name
        print(f"  Looking for match in object_data for: {repr(fname)}")

        if fname in object_data.index:
            print("  Match found in object_data.csv")
            row = object_data.loc[fname]

            if "hmin_pb" in row.index and not pd.isna(row["hmin_pb"]):
                hmin = float(row["hmin_pb"])
                print(f"    -> Using object-defined hmin = {hmin}")
            else:
                print("    (No hmin_pb defined, keeping default hmin)")

            if "h_hmax_pb" in row.index and not pd.isna(row["h_hmax_pb"]):
                hmax = float(row["h_hmax_pb"])
                print(f"    -> Using object-defined hmax = {hmax}")
            else:
                print("    (No h_hmax_pb defined, keeping default hmax)")
        else:
            print("  Warning: No match found in object_data.csv, using defaults.")

        print(f"  Final hmin = {hmin}, hmax = {hmax}")
        print(df.info())

        # Step 2: Flag physical bounds
        df_2, _ = flag_physical_bounds(df, hmin, hmax)

        # Step 3: Flag unrealistic steps
        df_3, _ = flag_unrealistic_step_change(df_2, max_up, max_down)

        # Step 4: Flag bottom-band values (v3)
        df_4, _ = flag_constant_head_periods(df_3, tconst_steps, flat_margin_m, min_band_m, hmin)

        # Step 5: Flag statistical outliers
        df_5, _ = flag_statistical_outliers(df_4)

        # Clean up temporary column
        df_final = df_5.drop(columns=["dH"], errors='ignore').copy()

        # Default approval state
        df_final["approved"] = 1

        # Save to output in per-origin folder
        output_dir = input_root / source_origin_stem / 'validated'
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / f'{csv_file.stem}.csv'

        # Preserve prior manual disapprovals (approved=0) on matching Time values
        if output_file.exists():
            try:
                existing = pd.read_csv(
                    output_file,
                    usecols=lambda c: c in ["Time", "approved"],
                    encoding="utf-8-sig",
                    encoding_errors="replace"
                )

                if {"Time", "approved"}.issubset(existing.columns):
                    existing["Time"] = pd.to_datetime(existing["Time"], errors="coerce")
                    zero_times = set(existing.loc[existing["approved"] == 0, "Time"].dropna())

                    if zero_times:
                        df_final["Time"] = pd.to_datetime(df_final["Time"], errors="coerce")
                        df_final.loc[df_final["Time"].isin(zero_times), "approved"] = 0
                        print(f"  Preserved {len(zero_times)} manual approved=0 timestamps")
            except Exception as merge_err:
                print(f"  Warning: could not merge existing approvals: {merge_err}")

        # Drop lineage columns from the validated output (they are re-derived
        # downstream from the file path when needed).
        df_final = df_final.drop(
            columns=["source_origin_stem", "source_series_file"],
            errors="ignore",
        )

        df_final.to_csv(output_file, index=False)

        # Count flagged rows
        flag_cols = [col for col in df_final.columns if col.startswith('v')]
        flagged_count = df_final[flag_cols].notna().any(axis=1).sum()

        results.append({
            'Filename': csv_file.name,
            'Origin': source_origin_stem,
            'Status': '[OK]',
            'Initial Rows': initial_rows,
            'Final Rows': len(df_final),
            'Flagged Points': flagged_count,
        })
        print(f"  [OK] Saved to {output_file}")
        print(f"    Initial: {initial_rows} rows -> Final: {len(df_final)} rows, {flagged_count} flagged\n")

    except Exception as e:
        print(f"  [ERR] Error: {str(e)}\n")
        failed_files.append(csv_file.name)
        results.append({
            'Filename': csv_file.name,
            'Origin': source_origin_stem,
            'Status': '[ERR]',
            'Initial Rows': 0,
            'Final Rows': 0,
            'Flagged Points': 0,
        })

# Summary
print("=" * 70)
print("BATCH PROCESSING COMPLETE")
print("=" * 70)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nTotal files processed: {len(results_df)}")
print(f"Successful: {len(results_df[results_df['Status'] == '[OK]'])}")
print(f"Failed: {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for fname in failed_files:
        print(f"  - {fname}")


object_data index sample (first 5 rows):
   'NL-2412417-HWM_B09-PB1_m_NAP.csv'
   'NL-2412417-HWM_B09-PB2_m_NAP.csv'
   'NL-2412417-HWM_B12-PB1_m_NAP.csv'
   'NL-2412417-HWM_B13-PB1_m_NAP.csv'
   'NL-2412417-HWM_B13-PB2_m_NAP.csv'

Found 281 CSV files to process

[1/281] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv (origin=83034-1)
 Length of df before cleaning: 4272
ℹ️  321 NaN head values in 'd:\Users\jvanruitenbeek\data_validation\output_data\wiertsema\83034-1\knmi\83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv' (kept, not dropped)
  Initial rows: 4272
  Looking for match in object_data for: '83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv'
  Match found in object_data.csv
    -> Using object-defined hmin = -2.79
    (No h_hmax_pb defined, keeping default hmax)
  Final hmin = -2.79, hmax = 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4272 entries, 0 to 4271
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  